# 🗄️ Synthetic Test Database — Northwind

Generate a synthetic replica from a relational database, with the goal of preserving realistic primary and foreign key relationships.

In this exercise we will be using the [Northwind](https://github.com/microsoft/sql-server-samples/tree/master/samples/databases/northwind-pubs) Database — a Microsoft's classic sample database for a fictional specialty food import/export company. Five interconnected tables: customers, orders, order lines, products, and categories.

In this notebook we will:
1. Read five of Northwind's tables and explore the relational schema
2. Define the schema — primary keys and foreign keys — as a structured configuration
3. Validate the schema with `MultiMetadata` and review any warnings
4. Train a `RelationalSynthesizer` to learn the joint, multi-table distribution
5. Generate a synthetic database replica with a configurable size multiplier
6. Understand how primary and foreign keys are handled during synthesis
7. Validate referential integrity in the synthetic output

## 1. Load the Northwind Database

Northwind is a classic relational database built around a food import/export business. It contains interconnected tables that model real-world complexity: customers place orders, orders contain line items for specific products, and products belong to categories.

In [1]:
import pandas as pd
from ydata.dataset import Dataset

CSV_FILES = {
    'categories':    '../data/northwind/categories.csv',
    'products':      '../data/northwind/products.csv',
    'customers':     '../data/northwind/customers.csv',
    'orders':        '../data/northwind/orders.csv',
    'order_details': '../data/northwind/order_details.csv',
}

datasets = {}

for name, path in CSV_FILES.items():
    df = pd.read_csv(path)
    datasets[name] = Dataset(df)

print('Loaded tables')
for name, df in datasets.items():
    print(f'  {name:<15} {df.nrows:>5,} rows  ×  {df.ncols:>2} columns')

Loaded tables
  categories          8 rows  ×   3 columns
  products           77 rows  ×  10 columns
  customers          93 rows  ×  11 columns
  orders            830 rows  ×  14 columns
  order_details   2,155 rows  ×   5 columns


/Users/fabianaclemente/Documents/github/ydata/src/ydatautils/ydata-utils/ydata/dataset/dataset.py:62: UserWarning: All the input Variable Types were set as `string` or `object`. It is recommend to revise the VariableType settings for optimal results.
  warnings.warn(


In [2]:
# Quick look at each table
for name, df in datasets.items():
    print(f'\n═══ {name} ═══')
    print(f'Columns: {list(df.columns)}')
    display(df.head(3))


═══ categories ═══
Columns: ['CategoryID', 'CategoryName', 'Description']


,CategoryID,CategoryName,Description
0,1,Beverages,"Soft drinks, coffees, teas, beers, and ales"
1,2,Condiments,"Sweet and savory sauces, relishes, spreads, an..."
2,3,Confections,"Desserts, candies, and sweet breads"



═══ products ═══
Columns: ['ProductID', 'ProductName', 'SupplierID', 'CategoryID', 'QuantityPerUnit', 'UnitPrice', 'UnitsInStock', 'UnitsOnOrder', 'ReorderLevel', 'Discontinued']


,ProductID,ProductName,SupplierID,CategoryID,QuantityPerUnit,UnitPrice,UnitsInStock,UnitsOnOrder,ReorderLevel,Discontinued
0,1,Chai,1,1,10 boxes x 20 bags,18.0,39,0,10,0
1,2,Chang,1,1,24 - 12 oz bottles,19.0,17,40,25,0
2,3,Aniseed Syrup,1,2,12 - 550 ml bottles,10.0,13,70,25,0



═══ customers ═══
Columns: ['CustomerID', 'CompanyName', 'ContactName', 'ContactTitle', 'Address', 'City', 'Region', 'PostalCode', 'Country', 'Phone', 'Fax']


,CustomerID,CompanyName,ContactName,ContactTitle,Address,City,Region,PostalCode,Country,Phone,Fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,<NA>,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitucin 2222,Mxico D.F.,<NA>,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquera,Antonio Moreno,Owner,Mataderos 2312,Mxico D.F.,<NA>,05023,Mexico,(5) 555-3932,<NA>



═══ orders ═══
Columns: ['OrderID', 'CustomerID', 'EmployeeID', 'OrderDate', 'RequiredDate', 'ShippedDate', 'ShipVia', 'Freight', 'ShipName', 'ShipAddress', 'ShipCity', 'ShipRegion', 'ShipPostalCode', 'ShipCountry']


,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry
0,10248,VINET,5,1996-07-04 00:00:00,1996-08-01 00:00:00,1996-07-16 00:00:00,3,32.38,Vins et alcools Chevalier,59 rue de l-Abbaye,Reims,<NA>,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00,1996-08-16 00:00:00,1996-07-10 00:00:00,1,11.61,Toms Spezialitten,Luisenstr. 48,Mnster,<NA>,44087,Germany
2,10250,HANAR,4,1996-07-08 00:00:00,1996-08-05 00:00:00,1996-07-12 00:00:00,2,65.83,Hanari Carnes,"Rua do Pao, 67",Rio de Janeiro,RJ,05454-876,Brazil



═══ order_details ═══
Columns: ['OrderID', 'ProductID', 'UnitPrice', 'Quantity', 'Discount']


,OrderID,ProductID,UnitPrice,Quantity,Discount
0,10248,11,14.0,12,0
1,10248,42,9.8,10,0
2,10248,72,34.8,5,0


## 2. The Relational Schema

Before we can synthesize a multi-table database, we need to tell the synthesizer how the tables relate to each other. This is not optional, without the schema, a model trained on individual tables independently will generate valid rows per table but destroy the links between them.

The Northwind schema we are working with:

```
Categories ──< Products ──< OrderDetails >── Orders >── Customers
    PK: CategoryID        PK: OrderDetailID   PK: OrderID   PK: CustomerID
                          FK: ProductID       FK: CustomerID
                          FK: OrderID
```

Each arrow represents a **one-to-many** relationship: one Category can have many Products, one Order can have many Order Details, and so on. The synthesizer must preserve all of these links in the output.

Note: If connecting directly to a database (like SQLServer, Oracle, Snowflake) the schema is infered directly from the database itself.

In [3]:
# Define the relational schema: primary keys and foreign key references
import pprint as pp
from yaml import safe_load

with open(f"data/northwind/schema.yml", "r") as f:
        schema = safe_load(f)
    
print('Schema defined ✓')
print(f'  Tables: {len(schema)}')
pp.pprint(schema)

Schema defined ✓
  Tables: 5
{'categories': {'primary_keys': ['CategoryID']},
 'customers': {'primary_keys': ['CustomerID']},
 'order_details': {'foreign_keys': [{'column': 'OrderID',
                                     'parent_column': 'OrderID',
                                     'parent_table': 'orders'},
                                    {'column': 'ProductID',
                                     'parent_column': 'ProductID',
                                     'parent_table': 'products'}],
                   'primary_keys': ['OrderID', 'ProductID']},
 'orders': {'foreign_keys': [{'column': 'CustomerID',
                              'parent_column': 'CustomerID',
                              'parent_table': 'customers'}],
            'primary_keys': ['OrderID']},
 'products': {'foreign_keys': [{'column': 'CategoryID',
                                'parent_column': 'CategoryID',
                                'parent_table': 'categories'}],
              'primary_keys': 

## 3. Configure the Synthesizer

In [7]:
from ydata.dataset import MultiDataset
from ydata.metadata import MultiMetadata

# Wrap all tables in a MultiDataset
database = MultiDataset(datasets, 
                             schema=schema)

# Build MultiMetadata — this validates the schema and infers column types
m = MultiMetadata(
    multiset=database,
)

print('MultiMetadata created ✓')

INFO: 2026-08-11 11:16:32,162 [MULTIMETADATA] - Calculating metadata for tables: {'customers', 'products', 'order_details', 'categories', 'orders'}.


Done:  80%|████████████████████████████       | 4/5 [00:00<00:00, 342.50step/s]

INFO: 2026-08-11 11:16:32,428 [MULTIMETADATA] - Initializing characteristics.
INFO: 2026-08-11 11:16:32,436 [MULTIMETADATA] - Validating schema.
INFO: 2026-08-11 11:16:32,436 [MULTIMETADATA] - Update relationship types.
MultiMetadata created ✓


In [8]:
# Review the metadata summary and any schema warnings
print('=== Metadata Summary ===')
print(m)

print('\n=== Schema Warnings ===')
# Warnings here point to potential issues: missing FK values, circular refeences,
# or referential integrity violations in the source.
m.warnings

=== Metadata Summary ===
MultiMetadata Summary 
 
Tables Summary 
Number of tables: 5 
 
      Table name  # cols  # nrows          Primary keys          Foreign keys PK characteristics                        FK characteristics Notes
0     categories       3        8          [CategoryID]                                     [id]                                                
1       products      10       77           [ProductID]          [CategoryID]               [id]                    {'CategoryID': ['id']}      
2      customers      11       93          [CustomerID]                                     [id]                                                
3         orders      14      830             [OrderID]          [CustomerID]               [id]                    {'CustomerID': ['id']}      
4  order_details       5     2155  [OrderID, ProductID]  [OrderID, ProductID]           [id, id]  {'OrderID': ['id'], 'ProductID': ['id']}      
 
Relations Summary 
Number of relations:

{}

## 4. Train the MultiTableSynthesizer

The `RelationalSynthesizer` learns the joint distribution across all tables simultaneously. It does not train one model per table and stitch them together, it learns how tables relate, so the generated data propagates realistic patterns from parent to child.

### How Primary and Foreign Keys Are Handled

This is one of the most important things to understand about relational synthesis.

**During training**, primary and foreign key columns are automatically anonymized. The model never learns the actual identifier values — `'ALFKI'`, `10248`, `11` — it learns the *structure*: how many orders a typical customer places, how many line items a typical order contains.

**During generation**, new primary keys are created from scratch. The foreign keys in child tables are then assigned from the pool of newly generated parent keys — so every `Orders.CustomerID` points to a `CustomerID` that exists in the synthetic `Customers` table.

The result: a database where no real identifiers appear, but every link that existed in the original structure is preserved in the synthetic output.

In [9]:
from ydata.synthesizers import MultiTableSynthesizer

synth = MultiTableSynthesizer()

synth.fit(X=database,
          metadata=m)

print('Training complete ✓')

INFO: 2026-08-11 11:16:37,313 [MULTITABLE] - Validate anonymization data
INFO: 2026-08-11 11:16:37,314 (1/5) - Fitting table: [customers]


/Users/fabianaclemente/Documents/github/ydata/src/ydatautils/ydata-utils/ydata/dataset/dataset.py:62: UserWarning: All the input Variable Types were set as `string` or `object`. It is recommend to revise the VariableType settings for optimal results.
  warnings.warn(
Done:  75%|██████████████████████████▎        | 3/4 [00:00<00:00, 142.75step/s]

Validate the segmentation strategy.
INFO: 2026-08-11 11:16:37,342 [SYNTHESIZER] - Number columns considered for synth: 11


INFO: 2026-08-11 11:16:37,685 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-11 11:16:37,687 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-11 11:16:37,692 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-11 11:16:37,693 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
INFO: 2026-08-11 11:16:37,749 (2/5) - Fitting table: [orders]
INFO: 2026-08-11 11:16:37,751 [SYNTHESIZER] - Encoding table orders


Done:  75%|███████████████████████████         | 3/4 [00:00<00:00, 92.40step/s]

...

Validate the segmentation strategy.
INFO: 2026-08-11 11:16:38,727 [SYNTHESIZER] - Number columns considered for synth: 14


INFO: 2026-08-11 11:16:39,008 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-11 11:16:39,009 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-11 11:16:39,011 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-11 11:16:39,011 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
INFO: 2026-08-11 11:16:39,053 [MULTITABLE] - Clearing tables from memory [{'categories'}]
INFO: 2026-08-11 11:16:39,053 (5/5) - Fitting table: [order_details]


Done:  75%|██████████████████████████▎        | 3/4 [00:00<00:00, 555.34step/s]

Validate the segmentation strategy.
INFO: 2026-08-11 11:16:39,063 [SYNTHESIZER] - Number columns considered for synth: 7


INFO: 2026-08-11 11:16:39,363 [SYNTHESIZER] - Starting the synthetic data modeling process over 1x1 blocks.
INFO: 2026-08-11 11:16:39,365 [SYNTHESIZER] - Preprocess segment
INFO: 2026-08-11 11:16:39,369 [SYNTHESIZER] - Synthesizer init.
INFO: 2026-08-11 11:16:39,370 [SYNTHESIZER] - Processing the data prior fitting the synthesizer.
INFO: 2026-08-11 11:16:39,463 [MULTITABLE] - Clearing tables from memory [{'products', 'orders'}]
Training complete ✓


## 5. Generate a Synthetic Database

Unlike single-table synthesis, where you specify an exact row count, relational synthesis uses a **size multiplier**.

| `sample_size` | Output size relative to original |
|---|---|
| `0.5` | Half as many rows in each table |
| `1.0` | Same size as the original database |
| `2.0` | Twice as many rows |

This design makes sense for relational data: you cannot simply set the order-line count without also controlling how many orders exist — the tables must grow together to preserve realistic cardinalities.

In [10]:
# Generate a synthetic database the same size as the original
synthetic_db = synth.sample(n_samples=1.0)

print('Generated synthetic database ✓')
print('\nTable sizes — original vs synthetic:')
print(f'{"Table":<15} {"Original":>10} {"Synthetic":>10}')
print('-' * 38)
for name, table in synthetic_db.items():
    orig_n = datasets[name].nrows
    synt_n = table.nrows
    print(f'{name:<15} {orig_n:>10,} {synt_n:>10,}')

INFO: 2026-08-11 11:16:39,468 (1/5) - Synthesizing table: customers
INFO: 2026-08-11 11:16:39,476 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-11 11:16:39,490 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-11 11:16:39,493 let write into the connector None
INFO: 2026-08-11 11:16:39,493 using kwargs {}
INFO: 2026-08-11 11:16:39,494 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-11 11:16:39,503 (2/5) - Synthesizing table: orders
INFO: 2026-08-11 11:16:39,510 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-11 11:16:39,558 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-11 11:16:39,564 let write into the connector None
INFO: 2026-08-11 11:16:39,564 using kwargs {}
INFO: 2026-08-11 11:16:39,566 [SYNTHESIZER] - Numerical clipping
INFO: 2026-08-11 11:16:39,577 (3/5) - Synthesizing table: categories
INFO: 2026-08-11 11:16:39,577 [SYNTHESIZER] - Start generating model samples.
INFO: 2026-08-11 11:16:39,580 [SYNTHESIZER] - Postprocessing.
INFO: 2026-08-11 11:16:39,58

In [11]:
## Show one original table and a synthetic

display(datasets["orders"].head())

display(synthetic_db["orders"].head())


,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry
0,10248,VINET,5,1996-07-04 00:00:00,1996-08-01 00:00:00,1996-07-16 00:00:00,3,32.38,Vins et alcools Chevalier,59 rue de l-Abbaye,Reims,<NA>,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00,1996-08-16 00:00:00,1996-07-10 00:00:00,1,11.61,Toms Spezialitten,Luisenstr. 48,Mnster,<NA>,44087,Germany
2,10250,HANAR,4,1996-07-08 00:00:00,1996-08-05 00:00:00,1996-07-12 00:00:00,2,65.83,Hanari Carnes,"Rua do Pao, 67",Rio de Janeiro,RJ,05454-876,Brazil
3,10251,VICTE,3,1996-07-08 00:00:00,1996-08-05 00:00:00,1996-07-15 00:00:00,1,41.34,Victuailles en stock,"2, rue du Commerce",Lyon,<NA>,69004,France
4,10252,SUPRD,4,1996-07-09 00:00:00,1996-08-06 00:00:00,1996-07-11 00:00:00,2,51.30,Suprmes dlices,"Boulevard Tirou, 255",Charleroi,<NA>,B-6000,Belgium


,OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry
0,1,1,2,1998-03-02 00:00:00,1998-03-10 00:00:00,1998-03-12 00:00:00,3,51.87,Wolski Zajazd,Obere Str. 57,Mxico D.F.,<NA>,05487-020,Mexico
1,2,1,1,1997-12-31 00:00:00,1997-09-19 00:00:00,1998-01-30 00:00:00,3,17.68,Wolski Zajazd,Obere Str. 57,Mxico D.F.,<NA>,05487-020,Mexico
2,3,1,2,1997-10-16 00:00:00,1997-09-19 00:00:00,1997-04-01 00:00:00,3,76.33,Wolski Zajazd,Obere Str. 57,Mxico D.F.,<NA>,05487-020,Mexico
3,4,1,1,1997-09-10 00:00:00,1997-09-19 00:00:00,1997-09-15 00:00:00,3,47.22,Wolski Zajazd,Obere Str. 57,Mxico D.F.,<NA>,05487-020,Mexico
4,5,1,2,1997-09-10 00:00:00,1997-04-23 00:00:00,1997-09-15 00:00:00,3,47.22,Wolski Zajazd,Obere Str. 57,Mxico D.F.,<NA>,05487-020,Mexico


## 7. Validate Referential integrity

**Referential integrity** means that every value in a foreign key column has a matching primary key in the parent table. If `Orders.CustomerID = 'XYZ99'` but there is no `Customers` row with `CustomerID = 'XYZ99'`, that order is an orphan — and any application that tries to look up the customer will fail.

The `RelationalSynthesizer` is designed to preserve referential integrity by construction, but validating it explicitly gives you the guarantee you need before deploying the synthetic database to a test environment.

In [12]:
from ydata.report import ReferentialIntegrityValidator  # (or wherever it's imported from)

validator = ReferentialIntegrityValidator(synthetic_data=synthetic_db, 
                                          metadata=m)
results = validator.validate()


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----

INFO: 2026-08-11 11:16:42,787 [REFERENTIAL_INTEGRITY] - Starting validation
INFO: 2026-08-11 11:16:42,792 [REFERENTIAL_INTEGRITY] - Validation complete. Overall score: 1.00


## ✅ Summary

In this notebook you:

1. **Trained a `MulitTable Synthetic data generator`** to learn the joint distribution across all five tables simultaneously
2. **Generated a synthetic database** using `sample_size=1.0`, and understood how the multiplier controls database scale
3. **Understood PK/FK handling**: real identifiers are anonymized during training; new keys are created and linked during generation
4. **Validated referential integrity** — both with the built-in validator and manually, confirming no orphan records

The key lesson: synthetic data for applications and APIs is not just about reproducing column statistics — it is about preserving the *structure* that makes the data usable. A synthetic database with broken foreign keys is not a test database; it is a bug factory!